In [1]:
import sys
import time
import warnings
from pathlib import Path

warnings.filterwarnings('ignore')

import joblib
import numpy as np
import pandas as pd
import seaborn as sns
from IPython.display import Markdown
import matplotlib.pyplot as plt

from sklearn.impute import SimpleImputer
from sklearn.compose import ColumnTransformer
from scipy.spatial.distance import cdist

from sklearn.cluster import KMeans
from sklearn.cluster import MiniBatchKMeans
from sklearn.cluster import BisectingKMeans
from sklearn.cluster import Birch
from sklearn.cluster import MeanShift
from sklearn.cluster import estimate_bandwidth
from sklearn.mixture import GaussianMixture
from sklearn.cluster import DBSCAN

from sklearn.pipeline import (
    Pipeline,
    TransformerMixin
)

from sklearn.preprocessing import (
    MinMaxScaler,
    OneHotEncoder
)

from sklearn.metrics import (
    silhouette_score,
    calinski_harabasz_score,
    davies_bouldin_score
)

from sklearn.model_selection import (
    train_test_split,
    cross_validate,
    StratifiedKFold,
    ShuffleSplit
)

from sklearn.feature_selection import VarianceThreshold
from sklearn.model_selection import GridSearchCV

# O notebook roda a partir de `notebooks/`; a raiz entra no path para que
# `import src...` funcione sem instalar o pacote.
RAIZ = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
if str(RAIZ) not in sys.path:
    sys.path.insert(0, str(RAIZ))

from src.data import dicionario as dic


In [2]:
data_path = Path('../data/processed/Telcon-Customer-Churn_tratado.csv')
dict_path = Path('../data/processed/dicionario.csv')

In [3]:
# Carregando a base de clientes
df = pd.read_csv(data_path)
display(Markdown(f'### Conjunto de dados — {len(df):,} clientes — {df.shape[1]} colunas'.replace(',', '.')))
display(df.head())

df_dict = (
    pd
    .read_csv(dict_path, sep=',')
    .assign(coluna=lambda d: d['variavel'])
)
display(Markdown('### Dicionário de dados'))
display(df_dict.head())

display(Markdown(
    'Tipos declarados: '
    + ' · '.join(f'**{tipo}** {n}' for tipo, n in df_dict['Tipo'].value_counts().items())
))

### Conjunto de dados — 7.043 clientes — 19 colunas

,gender,SeniorCitizen,Partner,Dependents,tenure,PhoneService,MultipleLines,InternetService,OnlineSecurity,OnlineBackup,DeviceProtection,TechSupport,StreamingTV,StreamingMovies,Contract,PaperlessBilling,PaymentMethod,MonthlyCharges,TotalCharges
0,Female,No,Yes,No,1,No,No,DSL,No,Yes,No,No,No,No,Month-to-month,Yes,Electronic check,29.85,29.85
1,Male,No,No,No,34,Yes,No,DSL,Yes,No,Yes,No,No,No,One year,No,Mailed check,56.95,1889.50
2,Male,No,No,No,2,Yes,No,DSL,Yes,Yes,No,No,No,No,Month-to-month,Yes,Mailed check,53.85,108.15
3,Male,No,No,No,45,No,No,DSL,Yes,No,Yes,Yes,No,No,One year,No,Bank transfer (automatic),42.30,1840.75
4,Female,No,No,No,2,Yes,No,Fiber optic,No,No,No,No,No,No,Month-to-month,Yes,Electronic check,70.70,151.65


### Dicionário de dados

,variavel,descricao,valores,Tipo,coluna
0,customerID,Unique identifier for each customer,"['7590-VHVEG', '3186-AJIEK']",Nominal,customerID
1,gender,Whether the customer is a male or a female,"['Female', 'Male']",Nominal,gender
2,SeniorCitizen,Whether the customer is a senior citizen or no...,"[0, 1]",Discreta,SeniorCitizen
3,Partner,Whether the customer has a partner or not (Yes...,"['Yes', 'No']",Nominal,Partner
4,Dependents,Whether the customer has dependents or not (Ye...,"['No', 'Yes']",Nominal,Dependents


Tipos declarados: **Nominal** 17 · **Discreta** 2 · **Contínua** 2

In [4]:
# Conferência dicionário x base tratada. O que não bate aqui vira decisão, não surpresa.
# (customerID e Churn são esperados na lista 'ausentes': foram retirados da base tratada.)
sem_mapa = df_dict.loc[df_dict['coluna'].isna(), 'variavel'].tolist()
fora_do_csv = [c for c in df_dict['coluna'].dropna() if c not in df.columns]
fora_do_dicionario = [c for c in df.columns if c not in set(df_dict['coluna'].dropna())]

display(Markdown(
    f'- variáveis do dicionário sem tradução: **{len(sem_mapa)}** {sem_mapa}\n'
    f'- no dicionário, ausentes do csv: **{len(fora_do_csv)}** {fora_do_csv}\n'
    f'- no csv, ausentes do dicionário (derivadas na curadoria): '
    f'**{len(fora_do_dicionario)}** {fora_do_dicionario}'
))


- variáveis do dicionário sem tradução: **0** []
- no dicionário, ausentes do csv: **2** ['customerID', 'Churn']
- no csv, ausentes do dicionário (derivadas na curadoria): **0** []

In [5]:
# Analisando o custo mensal dos serviços pagos pelos clientes
df.MonthlyCharges.describe()

count    7043.000000
mean       64.761692
std        30.090047
min        18.250000
25%        35.500000
50%        70.350000
75%        89.850000
max       118.750000
Name: MonthlyCharges, dtype: float64

In [ ]:
df.info()

In [7]:
useless_columns = [
    'customerID', 'Churn'
]


def colunas_do_tipo(*tipos):
    """Colunas da base curada que o dicionário declara com um dos `tipos`."""
    return [
        coluna
        for coluna in df_dict.query('Tipo in @tipos')['coluna'].dropna()
        if coluna in df.columns and coluna not in useless_columns
    ]


# Contínua e discreta
continuous_columns = colunas_do_tipo('Contínua', 'Discreta')

# AJUSTE (causa do erro "could not convert string to float: 'No'"):
# o dicionário declara SeniorCitizen como 'Discreta' (0/1), mas na base tratada
# a coluna virou texto ('No'/'Yes'). Com isso ela caía no MinMaxScaler, que só
# aceita números, e o fit quebrava. Em vez de confiar apenas no dicionário,
# conferimos o dtype real: toda coluna não numérica vai para as nominais
# (OneHotEncoder).
nao_numericas = [c for c in continuous_columns if not pd.api.types.is_numeric_dtype(df[c])]
continuous_columns = [c for c in continuous_columns if c not in nao_numericas]

# Ordinal & Nominal
ordinal_columns = colunas_do_tipo('Ordinal')

nominal_columns = colunas_do_tipo('Nominal') + nao_numericas

# AJUSTE: removidas as colunas derivadas de imóveis (area_varandas, tem_porao...), que não existem nesta base.

# Impressões
display(Markdown(
    f'- contínuas e discretas: **{len(continuous_columns)}**\n'
    f'- ordinais: **{len(ordinal_columns)}**\n'
    f'- nominais: **{len(nominal_columns)}**\n'
    f'- fora da modelagem: **{len(useless_columns)}** {useless_columns}'
))


- contínuas e discretas: **3**
- ordinais: **0**
- nominais: **16**
- fora da modelagem: **2** ['customerID', 'Churn']

In [8]:
continuous_preprocessor = Pipeline(steps=[
    ('normalization', MinMaxScaler()) # Normalização
])

ordinal_preprocessor = Pipeline(steps=[
    ('normalization', MinMaxScaler()) # Normalização — preserva a ordem e iguala o peso
])

nominal_preprocessor = Pipeline(steps=[
    ('encoder', OneHotEncoder(handle_unknown='ignore')), # Codificação de variáveis
])

preprocessor = ColumnTransformer(transformers=[
    ('continuous', continuous_preprocessor, continuous_columns),
    ('ordinal', ordinal_preprocessor, ordinal_columns),
    ('nominal', nominal_preprocessor, nominal_columns)
])


In [9]:
X = df
y = None # Aprendizagem não supervisionada: o rótulo não é utilizado

# A grade de `bandwidth` do mean shift não pode ser copiada do 07: ela depende da
# escala das distâncias, e a escala mudou junto com o espaço. Uma transformação
# única (não supervisionada, sem olhar o rótulo) dá a âncora.
X_ref = preprocessor.fit_transform(X)
bandwidth_ref = {
    q: estimate_bandwidth(X_ref, quantile=q, random_state=42)
    for q in (0.1, 0.2, 0.3, 0.5)
}
distancias = cdist(X_ref[:800], X_ref[:800])[np.triu_indices(800, k=1)]

display(Markdown(
    f'Espaço pré-processado: **{X_ref.shape[0]:,} × {X_ref.shape[1]}**  \n'.replace(',', '.')
    + f'Distâncias: p05 **{np.percentile(distancias, 5):.2f}** · '
    f'mediana **{np.median(distancias):.2f}** · p95 **{np.percentile(distancias, 95):.2f}**  \n'
    + '`estimate_bandwidth`: '
    + ' · '.join(f'q={q} → **{b:.2f}**' for q, b in bandwidth_ref.items())
))


Espaço pré-processado: **7.043 × 39**  
Distâncias: p05 **2.76** · mediana **4.01** · p95 **4.91**  
`estimate_bandwidth`: q=0.1 → **3.02** · q=0.2 → **3.34** · q=0.3 → **3.57** · q=0.5 → **3.93**

In [10]:
# Métricas de avaliação não supervisionadas (não dependem do rótulo y)
def _get_labels_and_features(estimator, X):
    """Obtém os dados transformados e os rótulos de cluster, tanto para o Pipeline
    completo (preprocessor + modelo) quanto para o modelo isolado (dentro do GridSearchCV)."""
    if isinstance(estimator, Pipeline):
        X_t = estimator[:-1].transform(X)
        labels = estimator[-1].predict(X_t)
    else:
        X_t = X
        labels = estimator.predict(X_t)
    return X_t, labels

def silhouette_scorer(estimator, X, y=None):
    X_t, labels = _get_labels_and_features(estimator, X)
    if len(np.unique(labels)) < 2:
        return -1.0
    return silhouette_score(X_t, labels)

scoring = {
    'silhouette': silhouette_scorer
}

bandwidths = [round(b, 2) for b in bandwidth_ref.values()]

models = [
    ("k-means", KMeans(n_init=10, random_state=42), {'n_clusters': [2, 3, 4, 5, 6, 8], 'init': ['k-means++', 'random']}),
    ("mini batch k-means", MiniBatchKMeans(n_init=10, random_state=42), {'n_clusters': [2, 3, 4, 5, 6, 8], 'batch_size': [64, 128, 256]}),
    ("bisecting k-means", BisectingKMeans(random_state=42), {'n_clusters': [2, 3, 4, 5, 6, 8], 'bisecting_strategy': ['biggest_inertia', 'largest_cluster']}),
    ("birch", Birch(), {'n_clusters': [2, 3, 4, 5, 6, 8], 'threshold': [0.1, 0.3, 0.5], 'branching_factor': [25, 50, 100]}),
    # ("mean shift", MeanShift(), {'bandwidth': bandwidths}),
    # ("gaussian mixture", GaussianMixture(random_state=42, max_iter=1000), {'n_components': [2, 3, 4, 5, 6, 8], 'covariance_type': ['full', 'tied', 'diag', 'spherical']}),
]


In [ ]:
def train_model(X, y=None, n_splits=10, final_results=None):  # Final result ajustado para None, par aevitar erros
    for model_name, model_obj, model_params in models:
        inicio = time.time()
        print(f'{model_name} run...', end=' ', flush=True)
        model_gs = GridSearchCV(
            model_obj,
            model_params,
            scoring=silhouette_scorer, # Seleção de hiperparâmetros pela métrica de silhueta
            cv=5
        )

        approach = Pipeline([
            ("preprocessor", preprocessor),
            ("model", model_gs)
        ])
        results = cross_validate(
            approach,
            X=X,
            y=y,
            scoring=scoring,
            cv=ShuffleSplit(n_splits=n_splits, test_size=.2, random_state=42),
            n_jobs=-1
        )
        results["name"] = [model_name] * n_splits
        print(f'{time.time() - inicio:.0f}s')

        if final_results is not None:
            for key, value in results.items():
                final_results[key] = np.append(final_results[key], value)
        else:
            final_results = results
    return final_results


In [12]:
final_results = train_model(X)

k-means run... 39s
mini batch k-means run... 19s
bisecting k-means run... 16s
birch run... 944s


In [13]:
# Métricas em que o menor valor é o melhor (tempos e Davies-Bouldin)
lower_is_better = ['fit_time', 'score_time', 'test_davies_bouldin']

def highlight_max(s, props=''):
    values = [float(value.split()[0]) for value in s.values[1:]]
    result = [''] * len(s.values)
    if s.values[0] in lower_is_better:
        result[np.argmin(values)+1] = props
    else:
        result[np.argmax(values)+1] = props
    return result

def get_winner(s):
    metric = s.values[0]
    values = [float(value.split()[0]) for value in s.values[1:]]
    models = results.columns[1:]

    if s.values[0] in lower_is_better:
        return models[np.argmin(values)]
    else:
        return models[np.argmax(values)]

results = (
    pd
    .DataFrame(final_results)
    .groupby(['name'])
    .agg([lambda x: f"{np.mean(x):.3f} ± {np.std(x):.3f}"])#
    .transpose()
    .reset_index()
    .rename(columns={"level_0": "score"})
    .drop(columns="level_1")
    # .set_index('score')
)
time_scores = ['fit_time', 'score_time']
winner = results.query('score not in @time_scores').apply(get_winner, axis=1).value_counts().index[0]
results.columns.name = ''
results = (
    results
    .style
    .hide(axis='index')
    .apply(highlight_max, props='color:white;background-color:gray', axis=1)
)
display(results)
display(Markdown(f'O melhor modelo é o : **{winner}**'))


score,birch,bisecting k-means,k-means,mini batch k-means
fit_time,938.194 ± 3.462,13.628 ± 1.388,30.739 ± 0.544,12.622 ± 0.373
score_time,0.125 ± 0.025,0.269 ± 0.150,0.155 ± 0.072,0.136 ± 0.025
test_silhouette,0.121 ± 0.012,0.149 ± 0.003,0.150 ± 0.003,0.150 ± 0.003


O melhor modelo é o : **k-means**

In [14]:
#Obtem o modelo e os parametros ganhadores
model_name, model, model_params  = [foo for foo in models if foo[0] == winner][0]

model_gs = GridSearchCV(model, model_params, scoring=silhouette_scorer)
approach = Pipeline([
    ("preprocessing", preprocessor),
    ("model", model_gs)
])
approach.fit(X) #Seleciona o approach

# Silhueta do modelo final em todo o conjunto de dados
X_t = approach[:-1].transform(X)
labels = approach[-1].predict(X_t)
display(Markdown(f'Silhueta do modelo final (**{model_name}**): **{silhouette_score(X_t, labels):.3f}**'))
display(Markdown(f'Melhores parâmetros: `{approach[-1].best_params_}`'))
display(Markdown(f'Grupos formados: **{len(np.unique(labels))}**'))


Silhueta do modelo final (**k-means**): **0.152**

Melhores parâmetros: `{'init': 'random', 'n_clusters': 2}`

Grupos formados: **2**

In [15]:
# Mapeando o perfil
# O perfil usa variáveis de clientes: tempo de casa, valores cobrados e
# a proporção de contratos mensais / fibra óptica / idosos em cada segmento.
perfil = (
    df
    .assign(
        segmento=labels,
        contrato_mensal=lambda d: d['Contract'].eq('Month-to-month'),
        fibra_optica=lambda d: d['InternetService'].eq('Fiber optic'),
        idoso=lambda d: d['SeniorCitizen'].eq('Yes'),
    )
    .groupby('segmento')
    .agg(
        clientes=('tenure', 'size'),
        tenure_mediano=('tenure', 'median'),
        mensalidade_mediana=('MonthlyCharges', 'median'),
        total_pago_mediano=('TotalCharges', 'median'),
        pct_contrato_mensal=('contrato_mensal', 'mean'),
        pct_fibra_optica=('fibra_optica', 'mean'),
        pct_idoso=('idoso', 'mean'),
    )
    .assign(participacao=lambda d: d['clientes'] / len(df))
    .sort_values('mensalidade_mediana', ascending=False)
)
display(Markdown('### Perfil dos segmentos'))
display(perfil.style.format({
    'tenure_mediano': '{:.0f}',
    'mensalidade_mediana': '${:,.2f}',
    'total_pago_mediano': '${:,.0f}',
    'pct_contrato_mensal': '{:.1%}',
    'pct_fibra_optica': '{:.1%}',
    'pct_idoso': '{:.1%}',
    'participacao': '{:.1%}',
}))


### Perfil dos segmentos

,clientes,tenure_mediano,mensalidade_mediana,total_pago_mediano,pct_contrato_mensal,pct_fibra_optica,pct_idoso,participacao
segmento,,,,,,,,
1,3136,50,$91.50,"$4,047",40.6%,62.5%,21.1%,44.5%
0,3907,14,$44.75,$559,66.6%,29.1%,12.3%,55.5%


In [16]:
# Salvando o Modelo.
modelo_path = Path(f'../models/model_bonus_telecom_{winner.replace(" ", "_").replace("-", "_")}.joblib')
joblib.dump(approach, modelo_path) # Salva o modelo em disco
display(Markdown(f'Modelo salvo em `{modelo_path}` ({modelo_path.stat().st_size / 1024:.0f} KB)'))


Modelo salvo em `..\models\model_bonus_telecom_k_means.joblib` (39 KB)

# Mapeando o perfil dos Clientes de Telecom

In [17]:
clientes = (
    df
    .assign(segmento=labels)
)

clientes.head()

,gender,SeniorCitizen,Partner,Dependents,tenure,PhoneService,MultipleLines,InternetService,OnlineSecurity,OnlineBackup,DeviceProtection,TechSupport,StreamingTV,StreamingMovies,Contract,PaperlessBilling,PaymentMethod,MonthlyCharges,TotalCharges,segmento
0,Female,No,Yes,No,1,No,No,DSL,No,Yes,No,No,No,No,Month-to-month,Yes,Electronic check,29.85,29.85,0
1,Male,No,No,No,34,Yes,No,DSL,Yes,No,Yes,No,No,No,One year,No,Mailed check,56.95,1889.50,0
2,Male,No,No,No,2,Yes,No,DSL,Yes,Yes,No,No,No,No,Month-to-month,Yes,Mailed check,53.85,108.15,0
3,Male,No,No,No,45,No,No,DSL,Yes,No,Yes,Yes,No,No,One year,No,Bank transfer (automatic),42.30,1840.75,0
4,Female,No,No,No,2,Yes,No,Fiber optic,No,No,No,No,No,No,Month-to-month,Yes,Electronic check,70.70,151.65,0
